In [0]:
use catalog identifier(:catalog);

In [0]:
CREATE TABLE IF NOT EXISTS silver.sessions (
  session_id       STRING,
  charger_id       STRING,
  member_id        STRING,
  started_at        TIMESTAMP,
  ended_at          TIMESTAMP,
  session_date      DATE,
  status            STRING,
  time_band         STRING,
  energy_kwh        DECIMAL(10,3),
  duration_minutes  INT,
  idle_minutes      INT,
  amount_paid       DECIMAL(10,2),
  updated_at        TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.session_faults (
  session_id   STRING,
  fault_seq    INT,
  fault_code   STRING
);

CREATE TABLE IF NOT EXISTS silver.quarantine_sessions (
  session_id      STRING,
  reject_reason   STRING,
  updated_at      TIMESTAMP,
  raw_record      STRING,
  quarantined_at  TIMESTAMP
);

In [0]:
CREATE OR REPLACE TEMP VIEW sessions_pipeline AS
WITH raw AS (
  SELECT
    *,
    to_json(struct(
      charger_id, ended_at, faults, idle_minutes, member_id,
      meter, payment, session_id, started_at, status, updated_at
    )) AS raw_record
  FROM bronze.sessions
),

std AS (
  SELECT
    
    NULLIF(UPPER(TRIM(session_id)), '') AS session_id,
    NULLIF(UPPER(TRIM(charger_id)), '') AS charger_id,
    NULLIF(UPPER(TRIM(member_id)), '') AS member_id,
   
    UPPER(TRIM(status)) AS status,
    
    COALESCE(
      try_to_timestamp(TRIM(started_at), 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(REPLACE(TRIM(started_at), 'T', ' '), 'yyyy-MM-dd HH:mm:ss')
    ) AS started_at_ts,
    COALESCE(
      try_to_timestamp(TRIM(ended_at), 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(REPLACE(TRIM(ended_at), 'T', ' '), 'yyyy-MM-dd HH:mm:ss')
    ) AS ended_at_ts,
 
    meter.start_kwh AS meter_start_kwh,
    meter.end_kwh   AS meter_end_kwh,
    
    COALESCE(CAST(idle_minutes AS INT), 0) AS idle_minutes,
   
    CAST(payment.amount_inr AS DECIMAL(10,2)) AS amount_paid,
    
    COALESCE(
      try_to_timestamp(TRIM(updated_at), 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(REPLACE(TRIM(updated_at), 'T', ' '), 'yyyy-MM-dd HH:mm:ss')
    ) AS updated_at_ts,
    faults,
    raw_record
  FROM raw
),

std2 AS (
  SELECT
    *,
    
    CAST(started_at_ts AS DATE) AS session_date,
    
    CASE WHEN meter_start_kwh IS NOT NULL AND meter_end_kwh IS NOT NULL
      THEN CAST(meter_end_kwh - meter_start_kwh AS DECIMAL(10,3))
      ELSE NULL
    END AS energy_kwh,
  
    CASE WHEN started_at_ts IS NOT NULL AND ended_at_ts IS NOT NULL
      THEN CAST(FLOOR((unix_seconds(ended_at_ts) - unix_seconds(started_at_ts)) / 60) AS INT)
      ELSE NULL
    END AS duration_minutes,
  
    CASE WHEN started_at_ts IS NOT NULL AND HOUR(started_at_ts) BETWEEN 18 AND 21
      THEN 'PEAK' ELSE 'OFF_PEAK'
    END AS time_band
  FROM std
),


dedup AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY session_id ORDER BY updated_at_ts DESC) AS rn
  FROM std2
  WHERE session_id IS NOT NULL
),

latest AS (
  SELECT * FROM dedup WHERE rn = 1
),


validated AS (
  SELECT
    l.*,
    c.charger_id AS matched_charger,
    m.member_id  AS matched_member
  FROM latest l
  LEFT JOIN silver.chargers c
    ON l.charger_id = c.charger_id
  LEFT JOIN (SELECT DISTINCT member_id FROM silver.members) m
    ON l.member_id = m.member_id
)

SELECT
  *,
  CASE
    WHEN charger_id IS NULL OR matched_charger IS NULL
      THEN 'UNKNOWN_CHARGER'                                    -- R1
    WHEN member_id IS NOT NULL AND matched_member IS NULL
      THEN 'UNKNOWN_MEMBER'                                     -- R2
    WHEN started_at_ts IS NULL
      THEN 'INVALID_TIMESTAMP'                                  -- R3
    WHEN status IN ('COMPLETED', 'FAILED')
         AND (ended_at_ts IS NULL OR ended_at_ts <= started_at_ts)
      THEN 'INVALID_DURATION'                                   -- R4
    WHEN status = 'COMPLETED'
         AND (energy_kwh IS NULL OR energy_kwh < 0)
      THEN 'INVALID_METER'                                      -- R5
    ELSE NULL
  END AS reject_reason
FROM validated;


CREATE OR REPLACE TEMP VIEW valid_faults AS
SELECT
  session_id,
  pos + 1 AS fault_seq,
  UPPER(TRIM(f.code)) AS fault_code
FROM sessions_pipeline
LATERAL VIEW posexplode(faults) AS pos, f
WHERE reject_reason IS NULL;

In [0]:

MERGE INTO silver.sessions AS t
USING (
  SELECT
    session_id,
    charger_id,
    member_id,
    started_at_ts  AS started_at,
    ended_at_ts    AS ended_at,
    session_date,
    status,
    time_band,
    energy_kwh,
    duration_minutes,
    idle_minutes,
    amount_paid,
    updated_at_ts  AS updated_at
  FROM sessions_pipeline
  WHERE reject_reason IS NULL
) AS s
ON t.session_id = s.session_id
WHEN MATCHED AND NOT (
      t.charger_id       <=> s.charger_id
  AND t.member_id        <=> s.member_id
  AND t.started_at       <=> s.started_at
  AND t.ended_at         <=> s.ended_at
  AND t.session_date     <=> s.session_date
  AND t.status           <=> s.status
  AND t.time_band        <=> s.time_band
  AND t.energy_kwh       <=> s.energy_kwh
  AND t.duration_minutes <=> s.duration_minutes
  AND t.idle_minutes     <=> s.idle_minutes
  AND t.amount_paid      <=> s.amount_paid
  AND t.updated_at       <=> s.updated_at
)
THEN UPDATE SET
  charger_id       = s.charger_id,
  member_id        = s.member_id,
  started_at       = s.started_at,
  ended_at         = s.ended_at,
  session_date     = s.session_date,
  status           = s.status,
  time_band        = s.time_band,
  energy_kwh       = s.energy_kwh,
  duration_minutes = s.duration_minutes,
  idle_minutes     = s.idle_minutes,
  amount_paid      = s.amount_paid,
  updated_at       = s.updated_at
WHEN NOT MATCHED THEN INSERT (
  session_id, charger_id, member_id, started_at, ended_at, session_date,
  status, time_band, energy_kwh, duration_minutes, idle_minutes, amount_paid, updated_at
)
VALUES (
  s.session_id, s.charger_id, s.member_id, s.started_at, s.ended_at, s.session_date,
  s.status, s.time_band, s.energy_kwh, s.duration_minutes, s.idle_minutes, s.amount_paid, s.updated_at
);


DELETE FROM silver.sessions
WHERE session_id IN (
  SELECT session_id FROM sessions_pipeline WHERE reject_reason IS NOT NULL
);

In [0]:


DELETE FROM silver.session_faults
WHERE session_id IN (
  SELECT session_id FROM sessions_pipeline WHERE reject_reason IS NOT NULL
);

DELETE FROM silver.session_faults AS t
WHERE t.session_id IN (
  SELECT session_id FROM sessions_pipeline WHERE reject_reason IS NULL
)
AND NOT EXISTS (
  SELECT 1 FROM valid_faults vf
  WHERE vf.session_id = t.session_id
  AND vf.fault_seq = t.fault_seq
);


MERGE INTO silver.session_faults AS t
USING valid_faults AS s
ON t.session_id = s.session_id AND t.fault_seq = s.fault_seq
WHEN MATCHED AND t.fault_code <> s.fault_code
  THEN UPDATE SET fault_code = s.fault_code
WHEN NOT MATCHED THEN INSERT (
  session_id, fault_seq, fault_code
)
VALUES (
  s.session_id, s.fault_seq, s.fault_code
);

In [0]:

MERGE INTO silver.quarantine_sessions AS t
USING (
  SELECT
    session_id,
    reject_reason,
    updated_at_ts AS updated_at,
    raw_record
  FROM sessions_pipeline
  WHERE reject_reason IS NOT NULL
) AS s
ON t.session_id = s.session_id
WHEN MATCHED AND NOT (
      t.reject_reason <=> s.reject_reason
  AND t.updated_at    <=> s.updated_at
  AND t.raw_record    <=> s.raw_record
)
THEN UPDATE SET
  reject_reason = s.reject_reason,
  updated_at    = s.updated_at,
  raw_record    = s.raw_record
WHEN NOT MATCHED THEN INSERT (
  session_id, reject_reason, updated_at, raw_record, quarantined_at
)
VALUES (
  s.session_id, s.reject_reason, s.updated_at, s.raw_record, current_timestamp
);

DELETE FROM silver.quarantine_sessions
WHERE session_id IN (
  SELECT session_id FROM sessions_pipeline WHERE reject_reason IS NULL
);